# 정적 웹페이지 데이터 전처리

- 실습 주제
이전 단계에서 data/interim 폴더에 저장된 첫 페이지 파싱 csv를 불러와
분석과 저장에 적합한 자료형과 구조로 변환한다.


# 패키지 불러오기

In [1]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd

# 프로젝트 루트와 데이터 경로 설정

현재 작업 경로부터 상위 폴더를 차례로 확인하여 `main.py`, `src`, `data`가 모두 존재하는 폴더를 프로젝트 루트로 선택한다.

In [2]:
Path.cwd()

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks/01_static')

In [3]:
even = [2, 4, 6, 8]
odd = [1, 3, 5, 7]
ex = [even, odd]
ex

[[2, 4, 6, 8], [1, 3, 5, 7]]

In [4]:
ex = [*even, *odd]
ex

[2, 4, 6, 8, 1, 3, 5, 7]

In [5]:
Path.cwd()

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks/01_static')

In [6]:
print(*Path.cwd().parents, sep='\n')

D:\AI\data_analytics\crawling\01-data-collection-pipeline\notebooks
D:\AI\data_analytics\crawling\01-data-collection-pipeline
D:\AI\data_analytics\crawling
D:\AI\data_analytics
D:\AI
D:\


In [7]:
Path.cwd().parents[0]

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks')

In [8]:
Path.cwd().parents[1]

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline')

In [9]:
Path.cwd().parents[2]

WindowsPath('D:/AI/data_analytics/crawling')

In [10]:
ex_path = Path.cwd()
ex_path

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks/01_static')

In [11]:
ex_path_all = [ex_path, *ex_path.parents]
ex_path_all

[WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks/01_static'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/notebooks'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline'),
 WindowsPath('D:/AI/data_analytics/crawling'),
 WindowsPath('D:/AI/data_analytics'),
 WindowsPath('D:/AI'),
 WindowsPath('D:/')]

In [12]:
(ex_path_all[0] / 'main.py').is_file()

False

In [13]:
(ex_path_all[2] / 'main.py')

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/main.py')

In [14]:
(ex_path_all[2] / 'main.py').is_file()

True

In [15]:
def find_project_root(start_path: Path | None = None) -> Path:
    """
    현재 위치로부터 상위 폴더를 탐색하여 프로젝트 루트를 반환한다.

    프로젝트 루트 판단 기준:
    - main.py 파일이 존재한다.
    - src 폴더가 존재한다.
    - data 폴더가 존재한다.

    Args:
        start_path:
            탐색을 시작할 경로
            값을 지정하지 않으면 현재 작업 경로를 사용한다.

    Returns:
        프로젝트 루트 경로

    Raises:
        FileNotFoundError:
            프로젝트 루트를 찾지 못한 경우    
    """
    current_path = (
        start_path if start_path is not None else Path.cwd()
    ).resolve()

    # print(f'{current_path = }')

    candidates = [current_path, *current_path.parents]

    for candidate in candidates:
        if (
            (candidate / 'main.py').is_file()
            and (candidate / 'src').is_dir()
            and (candidate /'data').is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        '프로젝트 루트를 찾지 못했습니다. '
        'main.py, src, data 구조를 확인하세요.'
    )    

In [16]:
# ex_path = Path('e:/aaa/bbb')
# find_project_root(ex_path)

In [17]:
find_project_root()

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline')

In [18]:
PROJECT_DIR = find_project_root()
PROJECT_DIR

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline')

In [19]:
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

print(f'현재 작업 경로 : {Path.cwd()}')
print(f'프로젝트 기준 경로 : {PROCESSED_DIR}')
print(f'중간 데이터 폴더 : {INTERIM_DIR}')
print(f'전처리 데이터 폴더 : {PROCESSED_DIR}')

현재 작업 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\notebooks\01_static
프로젝트 기준 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\processed
중간 데이터 폴더 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\interim
전처리 데이터 폴더 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\processed


# 최신 파싱 csv 파일 선택

`main.py`를 실행할 때마다 `data/interim` 폴더에 다음 형식의 파일이 생성된다.

```text
books_page_001_parsed_YYYYMMDD_HHMMSS.csv
```

파일명을 오름차순으로 정렬한 뒤 마지막 파일을 선택한다.

In [20]:
[*INTERIM_DIR.glob('*.csv')]

[WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_143406.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_175935.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_094913.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')]

In [21]:
sorted(INTERIM_DIR.glob('*.csv'))

[WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_143406.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_175935.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_094913.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')]

In [22]:
sorted(INTERIM_DIR.glob('*.csv'))[-1]

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')

In [23]:
def find_latest_parsed_csv(
    directory: Path,
    pattern: str = 'books_page_001_parsed_*.csv'
):
    """
    파일명 패턴과 일치하는 최신 파싱 csv를 반환한다.

    Args:
        directory:
            중간 csv 파일이 저장된 폴더

        pattern:
            검색할 파일명 패턴
    
    Returns:
        파일명 기준으로 가장 마지막에 있는 csv 파일 경로

    Raises:
        FileNotFoundError:
            폴더가 없거나 패턴에 맞는 csv가 없는 경우
    
    """

    if not directory.exists():
        raise FileNotFoundError(f'중간 데이터 폴더가 없습니다. {directory}')

    parsed_files = sorted(directory.glob(pattern))
    # print(parsed_files)

    if not parsed_files:
        raise FileNotFoundError(
            '전처리할 파싱 csv 파일이 없습니다. '
            '먼저 프로젝트 루트에서 main.py를 실행하세요'
        )

    return parsed_files[-1]   

In [24]:
find_latest_parsed_csv(INTERIM_DIR)

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')

In [25]:
latest_parsed_file = find_latest_parsed_csv(INTERIM_DIR)
print(f'❗ 선택한 파싱 csv 경로 : {latest_parsed_file}')
print(f'❗ 선택한 파싱 csv 파일명 : {latest_parsed_file.name}')

❗ 선택한 파싱 csv 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\interim\books_page_001_parsed_20260803_124432.csv
❗ 선택한 파싱 csv 파일명 : books_page_001_parsed_20260803_124432.csv


# 파싱 csv 불러오기

파싱 결과의 원본 형태를 그대로 확인하기 위해 우선 모든 컬럼을 문자열 자료형으로 불러온다.

평점, 가격, 재고 상태의 자료형 변환은 이후 전처리 단계에서 수행한다.

In [26]:
## string : Pandas nullable 문자열 dtype
## str    : Python 내장 문자열 클래스 
books_df = pd.read_csv(latest_parsed_file, dtype='string')

print(f'✔️행 수 : {len(books_df)}')
print(f'✔️컬럼 수 : {len(books_df.columns)}')

✔️행 수 : 20
✔️컬럼 수 : 7


In [27]:
books_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              20 non-null     string
 1   price_text         20 non-null     string
 2   availability_text  20 non-null     string
 3   rating_text        20 non-null     string
 4   rating             20 non-null     string
 5   detail_path        20 non-null     string
 6   detail_url         20 non-null     string
dtypes: string(7)
memory usage: 1.2 KB


In [28]:
ex = pd.DataFrame({
    '이름': ['홍길동', '이미자', None],
    '나이': [25, 52, 35],
})

ex

,이름,나이
0,홍길동,25
1,이미자,52
2,NaN,35


In [29]:
ex.이름.astype('str')

0    홍길동
1    이미자
2    NaN
Name: 이름, dtype: str

In [30]:
ex.이름[0]

'홍길동'

In [31]:
type(ex.이름[0])

str

In [32]:
ex.이름[2]

nan

In [33]:
type(ex.이름[2])

float

In [34]:
ex.이름.astype(str)[2]

nan

In [35]:
type(ex.이름.astype(str)[2])

float

In [36]:
ex.이름.astype('string')

0     홍길동
1     이미자
2    <NA>
Name: 이름, dtype: string

In [37]:
ex.이름.astype('string')[2]

<NA>

In [38]:
type(ex.이름.astype('string')[2])

pandas.api.typing.NAType

In [39]:
sorted(INTERIM_DIR.glob('*.csv'))

[WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_143406.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260731_175935.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_094913.csv'),
 WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')]

# 입력 데이터 기본 검증

전처리 전에 필요한 컬럼이 모두 존재하는지 확인한다.

In [40]:
books_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url'],
      dtype='str')

In [206]:
REQUIRED_INPUT_COLUMNS = {
    'title',
    'price_text',
    'availability_text',
    'rating_text',
    'rating',
    'detail_path',
    'detail_url',
}

missing_columns = [
   column 
    for column in REQUIRED_INPUT_COLUMNS
    if column not in books_df.columns
]

missing_columns

[]

In [42]:
if missing_columns:
    raise ValueError(f'입력 데이터의 필수 컬럼이 누락되었습니다. : {missing_columns}')

if books_df.empty:
    raise ValueError('입력 데이터가 비어 있습니다.')

print('입력 데이터 기본 검증 완료')

입력 데이터 기본 검증 완료


# 원본 파싱 데이터 보존

전처리 과정에서 값을 변경하기 전에 원본 DataFrame을 복사한다.

```python
processed_df = books_df.copy()
```

이후 `proceesed_df`를 수정해도 `books_df`는 변경되지 않는다.

In [43]:
processed_df = books_df.copy()

print(f'원본 행 수 : {len(books_df)}')
print(f'작업본 행 수 : {len(processed_df)}')

원본 행 수 : 20
작업본 행 수 : 20


# 문자열 컬럼 정리

문자열 컬럼의 앞뒤 공백을 제거하고
빈 문자열을 Pandas 결측값인 `pd.NA`로 변환한다.

파싱 단계의 원본 값을 확인할 수 있도록 다음 컬럼은 유지한다.

- `price_text`
- `availability_text`
- `rating_text`
- `detail_path`

In [44]:
len(processed_df.columns)

7

In [45]:
processed_df.head()

,title,price_text,availability_text,rating_text,rating,detail_path,detail_url
0,A Light in the Attic,£51.77,In stock,Three,3,catalogue/a-light-in-the-attic_1000/index.html,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,One,1,catalogue/tipping-the-velvet_999/index.html,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,One,1,catalogue/soumission_998/index.html,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,Four,4,catalogue/sharp-objects_997/index.html,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five,5,catalogue/sapiens-a-brief-history-of-humankind...,https://books.toscrape.com/catalogue/sapiens-a...


In [46]:
STRING_COLUMNS = [
    'title',
    'price_text',
    'availability_text',
    'rating_text',
    'detail_path',
    'detail_url',
]

for column in STRING_COLUMNS:
    processed_df[column] = (
        processed_df[column]
        .astype('string')
        .str
        .strip()
        .replace('', pd.NA)
    )

processed_df[STRING_COLUMNS].info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              20 non-null     string
 1   price_text         20 non-null     string
 2   availability_text  20 non-null     string
 3   rating_text        20 non-null     string
 4   detail_path        20 non-null     string
 5   detail_url         20 non-null     string
dtypes: string(6)
memory usage: 1.1 KB


# 가격 전처리

- 원본 값

```text
£51.77
```

- 전처리 결과
  
```text
price_text   -> £51.77
price        -> 51.77
```

가격 원본 문자열은 보존하고 숫자 부분만 추출하여
Pandas nullable 실수형인 `Float64`로 변환한다.

price 컬럼의 단위는 영국 파운드(GBP)이다.

In [47]:
processed_df.price_text[:2]

0    £51.77
1    £53.74
Name: price_text, dtype: string

In [48]:
processed_df.price_text.str.extract(r'(\d+\.\d+)', expand=False)

0     51.77
1     53.74
2     50.10
3     47.82
4     54.23
5     22.65
6     33.34
7     17.93
8     22.60
9     52.15
10    13.99
11    20.66
12    17.46
13    52.29
14    35.02
15    57.25
16    23.88
17    37.59
18    51.33
19    45.17
Name: price_text, dtype: string

## [함수 정의] parse_price : 가격 문자열에서 숫자 추출하여 Float64로 변환

In [49]:
def parse_price(price_series: pd.Series) -> pd.Series:
    """
    가격 문자열에서 숫자 부분을 추출하여 Float64로 변환한다.

    숫자로 변환할 수 없는 값은 errors='coerce'에 의해
    결측값으로 처리되며, 이후 검증 단계에서 확인한다.    
    """

    number_text = pd.Series(
        price_series
        .astype('string')
        .str
        .extract(r'(\d+\.\d+)', expand=False)       
    )

    # print(type(number_text))
    # print(number_text.dtype)

    return pd.to_numeric(number_text, errors='coerce').astype('Float64')

## price 컬럼 추가

In [50]:
processed_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url'],
      dtype='str')

In [51]:
processed_df['price'] = parse_price(processed_df.price_text)
processed_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url', 'price'],
      dtype='str')

In [52]:
processed_df[['price_text', 'price']].head()

,price_text,price
0,£51.77,51.77
1,£53.74,53.74
2,£50.10,50.1
3,£47.82,47.82
4,£54.23,54.23


In [53]:
print(f'price 자료형 : {processed_df['price'].dtype}')

price 자료형 : Float64


In [54]:
processed_df['price'].describe()

count         20.0
mean       38.0485
std      15.135231
min          13.99
25%        22.6375
50%          41.38
75%         51.865
max          57.25
Name: price, dtype: Float64

In [55]:
processed_df.describe()

,price
count,20.0
mean,38.0485
std,15.135231
min,13.99
25%,22.6375
50%,41.38
75%,51.865
max,57.25


# 재고 상태 전처리

재고 상태 문자열을 Pandas nullable Boolean으로 변환한다.

```text
In stock      -> True
Out of stock  -> False
알 수 없는 값 -> <NA>
```

알 수 없는 값을 `False`로 처리하지 않는 이유는
**재고 없음**과 **판단할 수 없음**이 서로 다른 상태이기 때문이다.

## [함수 정의] parse_availability : 재고 상태 문자열을 True, False, None으로 변환

In [59]:
' In stock   '.strip().lower()

'in stock'

In [60]:
' In stock   '.strip().casefold()

'in stock'

In [61]:
'ß'.lower()

'ß'

In [62]:
'ß'.casefold()

'ss'

In [63]:
def parse_availability(value: object) -> bool | None:
    """
    재고 상태 문자열을 True, False 또는 None으로 변환한다.    
    """

    if pd.isna(value):
        return None

    normalized = str(value).strip().casefold()

    if 'out of stock' in normalized:
        return False

    if 'in stock' in normalized:
        return True

    return None

## is_available 컬럼 추가

In [61]:
'ß'.lower()

'ß'

In [70]:
processed_df['is_available'] = (
    processed_df['availability_text']
    .map(parse_availability)
    .astype('boolean')
)

processed_df[['availability_text', 'is_available']].head()

,availability_text,is_available
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True


In [74]:
print(f'is_available 자료형 : {processed_df['is_available'].dtype}')

processed_df.is_available.value_counts(dropna=False)

is_available 자료형 : boolean


is_available
True    20
Name: count, dtype: Int64

# 평점 자료형 변환

파싱 csv는 모든 컬럼을 문자열로 읽었으므로 `rating`을 다시 숫자로 변환하고  
nullable 정수형인 `Int64`'로 지정한다

In [76]:
processed_df.rating.dtype

<StringDtype(storage='python', na_value=<NA>)>

In [80]:
processed_df['rating'] = (
    pd.to_numeric(
        processed_df['rating'],
        errors='coerce',
    ).astype('Int64')
)

processed_df[['rating_text', 'rating']].head()

,rating_text,rating
0,Three,3
1,One,1
2,One,1
3,Four,4
4,Five,5


In [81]:
processed_df.rating.dtype

Int64Dtype()

# 도서 식별자 생성

상세 URL의 마지막 부분에는 도서를 구분하는 숫자 식별값이 포함되어 있다.

```text
.../a-light-in-the-attic_1000/index.html	
```

이 값을 `book_id`로 추출한다.

> 이 규칙은 Books to Scrape의 URL 구조에 종속된다.

In [85]:
processed_df.detail_url[0]

'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'

In [88]:
processed_df.detail_url[1]

'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'

In [89]:
processed_df.detail_url[2]

'https://books.toscrape.com/catalogue/soumission_998/index.html'

In [90]:
processed_df.detail_url.str.extract(r'_(\d+)/index\.html$')

,0
0,1000
1,999
2,998
3,997
4,996
5,995
6,994
7,993
8,992
9,991


In [91]:
processed_df.detail_url.str.extract(r'_(\d+)/index\.html$', expand=False)

0     1000
1      999
2      998
3      997
4      996
5      995
6      994
7      993
8      992
9      991
10     990
11     989
12     988
13     987
14     986
15     985
16     984
17     983
18     982
19     981
Name: detail_url, dtype: string

In [92]:
processed_df['book_id'] = (
    processed_df['detail_url']
    .str
    .extract(r'_(\d+)/index\.html$', expand=False)
    .astype('string')
)

processed_df[['book_id', 'title', 'detail_url']].head()

,book_id,title,detail_url
0,1000,A Light in the Attic,https://books.toscrape.com/catalogue/a-light-i...
1,999,Tipping the Velvet,https://books.toscrape.com/catalogue/tipping-t...
2,998,Soumission,https://books.toscrape.com/catalogue/soumissio...
3,997,Sharp Objects,https://books.toscrape.com/catalogue/sharp-obj...
4,996,Sapiens: A Brief History of Humankind,https://books.toscrape.com/catalogue/sapiens-a...


# 파싱, 전처리 메타데이터 추가

- `parsed_at`

중간 파싱 csv가 생성된 시각이다.
입력된 csv 파일명은 `YYYYMMDD_HHMMSS` 값을 사용한다.

- `processed_at`

현재 노트북에서 전처리를 실행한 시각이다.

파일명에 저장된 시각에는 시간대 정보가 포함되어 있지 않다.  
따라서 두 시각은 데이터 이력을 확인하는 용도로 기록하고,  
서로 다른 실행환경에서는 시각의 선후 관계를 강제 검증하지 않는다. 

## [함수 정의] extract_file_timestamp() : 파일명에서 시각 추출

In [108]:
ex_html_path =  Path(r'D:\AI\data_analytics\crawling/01-data-collection-pipeline/data/raw/html/books_home_20260729_165900.html')
ex_html_path

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/raw/html/books_home_20260729_165900.html')

In [97]:
ex_match = re.search(r'(\d{8}_\d{6})', str(ex_html_path))
ex_match

<re.Match object; span=(62, 77), match='20260729_165900'>

In [98]:
## 전체 일치
ex_match.group()

'20260729_165900'

In [99]:
## 전체 괄호 그룹
ex_match.group(0)

'20260729_165900'

In [100]:
## 첫 번째 괄호 그룹
ex_match.group(1)

'20260729_165900'

In [102]:
## 두 번째 괄호 그룹 : 에러
# ex_match.group(2)

In [104]:
## [strptime]
## string parse time
## p: Parsing(파싱, 문자열 읽기)
## 문자열 형태의 날짜와 시간을 파싱하여 datetime 객체로 변환
## cf) strftime(string format time) : f는 format을 의미

ex_file_datetime = datetime.strptime(ex_match.group(1), '%Y%m%d_%H%M%S')
ex_file_datetime

datetime.datetime(2026, 7, 29, 16, 59)

In [105]:
pd.Timestamp(ex_file_datetime)

Timestamp('2026-07-29 16:59:00')

In [110]:
## 지정한 파일이나 디렉터리의 상세한 메타데이터
'''
st_size : 파일 바이트 크기
st_atime : 마지막으로 접근한 시간, Access Time(접근 시간)
st_mtime : 마지막으로 수정한 시간, Modification Time(수정 시간)
st_ctime : 메타데이터나 내용이 변경된 시간, Change Time(변경 시간)
'''

ex_html_path.stat()

os.stat_result(st_mode=33206, st_ino=2251799813716440, st_dev=1064113605835594275, st_nlink=1, st_uid=0, st_gid=0, st_size=51294, st_atime=1785812182, st_mtime=1785312163, st_ctime=1785312163)

In [111]:
ex_html_path.stat().st_mtime

1785312163.587883

In [112]:
datetime.fromtimestamp(ex_html_path.stat().st_mtime)

datetime.datetime(2026, 7, 29, 17, 2, 43, 587883)

In [114]:
def extract_file_timestamp(file_path: Path) -> pd.Timestamp:
    """
    파일명에서 YYYYMMDD_HHMMSS 형식의 시각을 추출한다.

    파일명에서 시각을 찾지 못하면 파일 수정 시각을 사용한다.    
    """
    match = re.search(r'(\d{8}_\d{6})', str(file_path))

    if match:
        file_datetime = datetime.strptime(match.group(1), '%Y%m%d_%H%M%S')
        return pd.Timestamp(file_datetime)

    return pd.Timestamp(datetime.fromtimestamp(file_path.stat().st_mtime))

In [115]:
latest_parsed_file

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/interim/books_page_001_parsed_20260803_124432.csv')

In [146]:
BASE_URL = 'https://books.toscrape.com/'

parsed_at = extract_file_timestamp(latest_parsed_file)
parsed_at

processed_at = pd.Timestamp.now().floor('s')
processed_at

processed_df['source_site'] = pd.Series('Books to Scape', index=processed_df.index, dtype='string')
processed_df['source_url'] = pd.Series(BASE_URL, index=processed_df.index, dtype='string')
processed_df['source_page'] = pd.Series(1, index=processed_df.index, dtype='Int64')

processed_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url', 'price', 'is_available', 'book_id',
       'source_site', 'source_url', 'source_page', 'parsed_at'],
      dtype='str')

In [171]:
latest_parsed_file.name

'books_page_001_parsed_20260803_124432.csv'

In [174]:
processed_df['source_file'] = pd.Series(
    latest_parsed_file.name,
    index=processed_df.index,
    dtype='string'
)

processed_df.source_file

0     books_page_001_parsed_20260803_124432.csv
1     books_page_001_parsed_20260803_124432.csv
2     books_page_001_parsed_20260803_124432.csv
3     books_page_001_parsed_20260803_124432.csv
4     books_page_001_parsed_20260803_124432.csv
5     books_page_001_parsed_20260803_124432.csv
6     books_page_001_parsed_20260803_124432.csv
7     books_page_001_parsed_20260803_124432.csv
8     books_page_001_parsed_20260803_124432.csv
9     books_page_001_parsed_20260803_124432.csv
10    books_page_001_parsed_20260803_124432.csv
11    books_page_001_parsed_20260803_124432.csv
12    books_page_001_parsed_20260803_124432.csv
13    books_page_001_parsed_20260803_124432.csv
14    books_page_001_parsed_20260803_124432.csv
15    books_page_001_parsed_20260803_124432.csv
16    books_page_001_parsed_20260803_124432.csv
17    books_page_001_parsed_20260803_124432.csv
18    books_page_001_parsed_20260803_124432.csv
19    books_page_001_parsed_20260803_124432.csv
Name: source_file, dtype: string

In [140]:
parsed_at

Timestamp('2026-08-03 12:44:32')

In [147]:
processed_at

Timestamp('2026-08-04 12:47:26')

In [144]:
processed_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url', 'price', 'is_available', 'book_id',
       'source_site', 'source_url', 'source_page'],
      dtype='str')

In [145]:
processed_df['parsed_at'] = parsed_at
processed_df[['title', 'parsed_at']]

,title,parsed_at
0,A Light in the Attic,2026-08-03 12:44:32
1,Tipping the Velvet,2026-08-03 12:44:32
2,Soumission,2026-08-03 12:44:32
3,Sharp Objects,2026-08-03 12:44:32
4,Sapiens: A Brief History of Humankind,2026-08-03 12:44:32
5,The Requiem Red,2026-08-03 12:44:32
6,The Dirty Little Secrets of Getting Your Dream...,2026-08-03 12:44:32
7,The Coming Woman: A Novel Based on the Life of...,2026-08-03 12:44:32
8,The Boys in the Boat: Nine Americans and Their...,2026-08-03 12:44:32
9,The Black Maria,2026-08-03 12:44:32


In [149]:
processed_df['processed_at'] = processed_at
processed_df[['title', 'parsed_at', 'processed_at']]

,title,parsed_at,processed_at
0,A Light in the Attic,2026-08-03 12:44:32,2026-08-04 12:47:26
1,Tipping the Velvet,2026-08-03 12:44:32,2026-08-04 12:47:26
2,Soumission,2026-08-03 12:44:32,2026-08-04 12:47:26
3,Sharp Objects,2026-08-03 12:44:32,2026-08-04 12:47:26
4,Sapiens: A Brief History of Humankind,2026-08-03 12:44:32,2026-08-04 12:47:26
5,The Requiem Red,2026-08-03 12:44:32,2026-08-04 12:47:26
6,The Dirty Little Secrets of Getting Your Dream...,2026-08-03 12:44:32,2026-08-04 12:47:26
7,The Coming Woman: A Novel Based on the Life of...,2026-08-03 12:44:32,2026-08-04 12:47:26
8,The Boys in the Boat: Nine Americans and Their...,2026-08-03 12:44:32,2026-08-04 12:47:26
9,The Black Maria,2026-08-03 12:44:32,2026-08-04 12:47:26


In [150]:
print(f'파싱 csv 생성 시각 : {parsed_at}')
print(f'전처리 실행 시각 : {processed_at}')

파싱 csv 생성 시각 : 2026-08-03 12:44:32
전처리 실행 시각 : 2026-08-04 12:47:26


# 중복 데이터 확인 및 제거

한 페이지에서 `detail_url`은 각 도서를 구분하는 기준으로 사용한다.

중복 후보를 먼저 확인한 뒤,  
같은 상세 URL이 여러 번 존재하면 마지막 행을 남기고 제거한다.

In [154]:
processed_df.duplicated(subset=['detail_url'], keep=False)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
dtype: bool

In [155]:
duplicate_mask = processed_df.duplicated(subset=['detail_url'], keep=False)
duplicate_mask

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
dtype: bool

In [161]:
## book_id, title, detail_url

duplicate_candidates_df = (
    processed_df
    .loc[duplicate_mask, ['book_id', 'title', 'detail_url']]
    .sort_values('detail_url')
)

print(f'중복 후보 행 수 : {len(duplicate_candidates_df)}')

중복 후보 행 수 : 0


In [164]:
## 중복 제거 전
row_count_before = len(processed_df)

processed_df = (
    processed_df
        .drop_duplicates(subset='detail_url', keep='last')
        .reset_index(drop=True)
)

processed_df.shape

(20, 15)

In [165]:
row_count_after = len(processed_df)
row_count_after

20

In [166]:
removed_duplicate_count = row_count_before - row_count_after
removed_duplicate_count

0

In [167]:
print(f'중복 제거 전 : {row_count_before}')
print(f'중복 제거 후 : {row_count_after}')
print(f'제거된 중복 : {removed_duplicate_count}')

중복 제거 전 : 20
중복 제거 후 : 20
제거된 중복 : 0


# 컬럼 순서 정리

분석에 자주 사용하는 컬럼을 앞에 배치하고,  
원본 추적 컬럼과 메타데이터는 뒤에 배치한다.

In [175]:
processed_df.columns

Index(['title', 'price_text', 'availability_text', 'rating_text', 'rating',
       'detail_path', 'detail_url', 'price', 'is_available', 'book_id',
       'source_site', 'source_url', 'source_page', 'parsed_at', 'processed_at',
       'source_file'],
      dtype='str')

In [176]:
COLUMN_ORDER = [
    'book_id',
    'title',
    'price',
    'rating',
    'is_available',
    'detail_url',
    'source_site',
    'source_url',
    'source_page', 
    'parsed_at',
    'processed_at',
    'source_file',
    'price_text',
    'availability_text',
    'rating_text',
    'detail_path',
]

In [177]:
processed_df = processed_df[COLUMN_ORDER]
processed_df.columns

Index(['book_id', 'title', 'price', 'rating', 'is_available', 'detail_url',
       'source_site', 'source_url', 'source_page', 'parsed_at', 'processed_at',
       'source_file', 'price_text', 'availability_text', 'rating_text',
       'detail_path'],
      dtype='str')

In [178]:
processed_df[:2]

,book_id,title,price,rating,is_available,detail_url,source_site,source_url,source_page,parsed_at,processed_at,source_file,price_text,availability_text,rating_text,detail_path
0,1000,A Light in the Attic,51.77,3,True,https://books.toscrape.com/catalogue/a-light-i...,Books to Scape,https://books.toscrape.com/,1,2026-08-03 12:44:32,2026-08-04 12:47:26,books_page_001_parsed_20260803_124432.csv,£51.77,In stock,Three,catalogue/a-light-in-the-attic_1000/index.html
1,999,Tipping the Velvet,53.74,1,True,https://books.toscrape.com/catalogue/tipping-t...,Books to Scape,https://books.toscrape.com/,1,2026-08-03 12:44:32,2026-08-04 12:47:26,books_page_001_parsed_20260803_124432.csv,£53.74,In stock,One,catalogue/tipping-the-velvet_999/index.html


# 전처리 결과 검증

검증 항목은 다음과 같다.

1. 전처리 결과가 비어 있지 않은가?
2. 필수 컬럼이 모두 존재하는가?
3. 주요 분석 컬럼에 결측값이 없는가?
4. 가격이 0보다 큰가?
5. 평점이 1부터 5 사이인가?
6. 상세 URL이 기준 사이트의 절대 URL인가?
7. 상세 URL 중복이 제거되었는가?

`parsed_at`과 `processed_at`은 기록 여부만 확인한다.
파일명 시각에는 시간대 정보가 없으므로 서로 다른 실행환경에서  
두 시각의 선후 관계를 검증하지 않는다.

In [180]:
if {2}:
    print('집합에 요소가 있다')

집합에 요소가 있다


In [181]:
        required_not_null = [
            'book_id', 
            'title', 
            'price', 
            'rating', 
            'is_available',
            'detail_url',
            'parsed_at',
            'processed_at',
        ]

In [189]:
ex_null_counts = processed_df[required_not_null].isna().sum()
ex_null_counts[ex_null_counts > 0]

Series([], dtype: int64)

In [190]:
processed_df.price.isna()

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: price, dtype: bool

In [191]:
processed_df.price <= 0

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: price, dtype: boolean

In [193]:
(processed_df.price.isna() | (processed_df.price <= 0)).sum()

np.int64(0)

In [196]:
~processed_df.rating.between(1, 5)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: rating, dtype: boolean

In [198]:
(processed_df.rating.isna() | ~processed_df.rating.between(1, 5)).sum()

np.int64(0)

In [203]:
~processed_df.detail_url.fillna('').str.startswith(BASE_URL)

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: detail_url, dtype: boolean

In [201]:
BASE_URL

'https://books.toscrape.com/'

In [210]:
def validate_processed_books(
    df: pd.DataFrame,
    base_url: str,
    required_columns: set,
) -> dict[str, int]:
    """
    전처리된 도서 데이터의 품질 규칙을 검증한다.

    Returns:
        검증 요약 정보

    Raises:
        ValueError:
            하나 이상의 검증 규칙을 통과하지 못한 경우    
    """

    errors: list[str] = list()

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        errors.append(f'필수 컬럼 누락 : {sorted(missing_columns)}')

    if not missing_columns and not df.empty:
        required_not_null = [
            'book_id', 
            'title', 
            'price', 
            'rating', 
            'is_available',
            'detail_url',
            'parsed_at',
            'processed_at',
        ]

        null_counts = df[required_not_null].isna().sum()
        invalid_null_counts = null_counts[null_counts > 0]

        if not invalid_null_counts.empty:
            errors.append(f'필수값 결측 :\n{invalid_null_counts.to_string()}')

        invalid_price_count = (df['price'].isna() | (df['price'] <= 0)).sum()

        if invalid_price_count:
            errors.append(f'유효하지 않은 가격 : {invalid_price_count} 건')

        invalid_rating_count = (df['rating'].isna() | ~df['rating'].between(1, 5)).sum()

        if invalid_rating_count:
            errors.append(f'유효하지 않은 평점 : {invalid_rating_count} 건')
        
        invalid_url_count = (
            df['detail_url'].isna() 
            | ~df['detail_url'].fillna('').str.startswith(base_url)
        ).sum()

        if invalid_url_count:
            errors.append(f'유효하지 않은 상세 URL : {invalid_url_count} 건')

        duplicate_url_count = df['detail_url'].duplicated(keep=False).sum()

        if duplicate_url_count:
            errors.append(f'중복 상세 URL : {duplicate_url_count} 건')

    if errors:
        raise ValueError('전처리 데이터 검증 실패\n' + '\n\n'.join(errors))
        
    return {
        'row_count': len(df),
        'column_count': len(df.columns),
        'duplicate_url_count': df['detail_url'].duplicated(keep=False).sum(),
        'null_count': df.isna().sum().sum()
    }

In [212]:
validation_summary = validate_processed_books(processed_df, BASE_URL, REQUIRED_INPUT_COLUMNS)

print('전처리 데이터 검증 완료')
validation_summary

전처리 데이터 검증 완료


{'row_count': 20,
 'column_count': 16,
 'duplicate_url_count': np.int64(0),
 'null_count': np.int64(0)}

# 전처리 csv 원자적 저장

일반적으로 csv 저장은 저장 도중 프로그램이 종료되면  
불완전한 파일로 남을 수 있다.

저장은 다음 순서로 처리한다.
```
임시 파일에 저장
-> 저장 성공
-> 최종 파일명으로 교체
```

## [함수 정의] ensure_directory() : 폴더가 없으면 생성하고, 폴더 경로를 반환

In [213]:
def ensure_directory(directory: Path) -> Path:
    """
    폴더가 없으면 생성하고, 폴더 경로를 반환한다.
    """

    directory.mkdir(parents=True, exist_ok=True)

    return directory


## [함수 정의] save_csv_atomically() : DataFrame을 입시 csv에 저장한 후 최종 파일로 교체

In [214]:
def save_csv_atomically(
    df: pd.DataFrame,
    file_path: Path,
) -> Path:
    """
    DataFrame을 임시 csv에 저장한 후, 최종 파일로 교체한다.

    Args:
        df:
            저장할 DataFrame

        file_path:
            최종 csv 파일 경로

    Returns:
        저장된 최종 csv 파일 경로
    """

    ensure_directory(file_path.parent)
    temp_path = file_path.with_suffix('.tmp.csv')

    try:
        df.to_csv(
            temp_path, 
            index=False, 
            encoding='utf-8-sig',
            date_format='%Y-%m-%d %H:%M:%S'
        )
        temp_path.replace(file_path)
    except Exception:
        if temp_path.exists():
            temp_path.unlink()
        raise
            
    return file_path    

## 저장 파일명

입력 파싱 csv의 시각을 유지하며,  
중간 데이터와 전처리 결과의 연결 관계를 표시한다.

```text
중간 csv
books_page_001_parsed_20260803_124432.csv

전처리 csv
books_page_001_processed_20260803_124432.csv
```

In [216]:
parsed_at

Timestamp('2026-08-03 12:44:32')

In [218]:
parsed_timestamp = parsed_at.strftime('%Y%m%d_%H%M%S')
parsed_timestamp

'20260803_124432'

In [225]:
output_file = PROCESSED_DIR / f'books_page_001_processed_{parsed_timestamp}.csv'
output_file

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/processed/books_page_001_processed_20260803_124432.csv')

In [226]:
output_file.parent

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/processed')

In [227]:
output_file.with_suffix('.tmp.csv')

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/processed/books_page_001_processed_20260803_124432.tmp.csv')

In [ ]:
ensure_directory(output_file)

In [228]:
output_file

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/processed/books_page_001_processed_20260803_124432.csv')

In [230]:
saved_file = save_csv_atomically(processed_df, output_file)
saved_file

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/processed/books_page_001_processed_20260803_124432.csv')

# 저장 결과 다시 읽어 검증

저장된 csv를 다시 읽어 다음 항목을 확인한다.

- 저장 전후 행 수
- 저장 전후 컬럼 순서

In [231]:
saved_df = pd.read_csv(saved_file)
saved_df.shape

(20, 16)

In [233]:
if len(saved_df) != len(processed_df):
    raise ValueError('csv 저장 전후의 행 수가 다릅니다.')

if list(saved_df.columns) != list(processed_df.columns):
    raise ValueError('csv 저장 전후의 컬럼 순서가 다릅니다.')

In [234]:
saved_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   book_id            20 non-null     int64  
 1   title              20 non-null     str    
 2   price              20 non-null     float64
 3   rating             20 non-null     int64  
 4   is_available       20 non-null     bool   
 5   detail_url         20 non-null     str    
 6   source_site        20 non-null     str    
 7   source_url         20 non-null     str    
 8   source_page        20 non-null     int64  
 9   parsed_at          20 non-null     str    
 10  processed_at       20 non-null     str    
 11  source_file        20 non-null     str    
 12  price_text         20 non-null     str    
 13  availability_text  20 non-null     str    
 14  rating_text        20 non-null     str    
 15  detail_path        20 non-null     str    
dtypes: bool(1), float64(1), int64(3), str(1

In [235]:
processed_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   book_id            20 non-null     string        
 1   title              20 non-null     string        
 2   price              20 non-null     Float64       
 3   rating             20 non-null     Int64         
 4   is_available       20 non-null     boolean       
 5   detail_url         20 non-null     string        
 6   source_site        20 non-null     string        
 7   source_url         20 non-null     string        
 8   source_page        20 non-null     Int64         
 9   parsed_at          20 non-null     datetime64[us]
 10  processed_at       20 non-null     datetime64[us]
 11  source_file        20 non-null     string        
 12  price_text         20 non-null     string        
 13  availability_text  20 non-null     string        
 14  rating_text        20 n

# 전처리 결과 요약

In [236]:
print('=' * 70)
print('정적 웹페이지 전처리 결과')
print('=' * 70)

print(f'입력 파싱 csv : {latest_parsed_file.name}')
print(f'파싱 데이터 수 : {len(books_df)}')
print(f'제거된 중복 수 : {removed_duplicate_count}')
print(f'가격 변환 실패 수 : {processed_df.price.isna().sum()}')
print(f'재고 변환 실패 수 : {processed_df.is_available.isna().sum()}')
print(f'식별자 추출 실패 수 : {processed_df.book_id.isna().sum()}')
print(f'재고 보유 도서 수 : {processed_df.is_available.sum()}')
print(f'파싱 csv 생성 시각 : {parsed_at}')
print(f'전처리 실행 시각 : {processed_at}')
print(f'전처리 csv 저장 경로 : {saved_file}')

정적 웹페이지 전처리 결과
입력 파싱 csv : books_page_001_parsed_20260803_124432.csv
파싱 데이터 수 : 20
제거된 중복 수 : 0
가격 변환 실패 수 : 0
재고 변환 실패 수 : 0
식별자 추출 실패 수 : 0
재고 보유 도서 수 : 20
파싱 csv 생성 시각 : 2026-08-03 12:44:32
전처리 실행 시각 : 2026-08-04 12:47:26
전처리 csv 저장 경로 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\processed\books_page_001_processed_20260803_124432.csv
